# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamKottish/FlyRankML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import sys
import subprocess
from pathlib import Path
import pandas as pd

# Find the starter CSV. If it is not available in Colab,
# clone the starter repository.
possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"),
]

DATA_PATH = next((p for p in possible_paths if p.exists()), None)

if DATA_PATH is None and "google.colab" in sys.modules:
    repo_dir = Path("/content/flyrank-ml-internship-starter")

    if not repo_dir.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
                str(repo_dir),
            ],
            check=True,
        )

    DATA_PATH = repo_dir / "data/raw/content_refresh_anonymized.csv"

assert DATA_PATH.exists(), "Starter CSV not found."

df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded: {len(df):,} rows × {len(df.columns)} columns")
print(f"Clients: {df['client_id'].nunique():,}")

Dataset loaded: 30,000 rows × 44 columns
Clients: 32


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


Task type: Ranking / scoring**

My lane is CTR / Engagement Opportunity Scoring. The goal is not simply to classify pages as good or bad. The useful decision is to rank content items so that an SEO specialist or content editor knows which pages should be reviewed first.

Each content item will receive an opportunity score based on observed signals such as impressions, CTR relative to similar ranking positions, sessions, engagement, content age, and other safe content features. The final output will be a ranked review queue, with the highest-opportunity pages appearing first.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

ranking_columns = [
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "position_tier",
    "sessions_90d",
    "engagement_rate",
]

missing = [col for col in ranking_columns if col not in df.columns]

assert not missing, f"Missing columns: {missing}"

print("Task type: Ranking / scoring")
print("Output: one opportunity score per content item")
print("Required ranking signals are available.")

Task type: Ranking / scoring
Output: one opportunity score per content item
Required ranking signals are available.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*


For this early stage, I will use a provisional opportunity proxy, not a final ground-truth target.

A page is considered a current review opportunity when it has enough evidence and shows either:

- CTR meaningfully below the median CTR of pages in the same position tier, or
- meaningful session volume but weak measured engagement.

This proxy is built from observed measurements, but the thresholds are still rules chosen for the analysis, so I will not treat it as proof that a page needs editing.

For the later ML stage, I would prefer an observed future outcome from the warehouse data, such as whether a page continues to underperform or improves in a later time window. The feature window must come before the outcome window to avoid leakage.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
MIN_IMPRESSIONS = 500
MIN_SESSIONS = 50
MIN_CTR_GAP_PP = 0.10
LOW_ENGAGEMENT_PCT = 30

# Valid CTR analysis rows
proxy_df = df[df["avg_position"] > 0].copy()

# Expected CTR depends strongly on ranking position,
# so compare only within the same position tier.
proxy_df["tier_median_ctr"] = (
    proxy_df.groupby("position_tier")["ctr"]
    .transform("median")
)

proxy_df["ctr_gap_pp"] = (
    proxy_df["tier_median_ctr"] - proxy_df["ctr"]
)

proxy_df["ctr_opportunity"] = (
    (proxy_df["impressions_90d"] >= MIN_IMPRESSIONS)
    & (proxy_df["ctr_gap_pp"] > MIN_CTR_GAP_PP)
)

proxy_df["engagement_opportunity"] = (
    (proxy_df["sessions_90d"] >= MIN_SESSIONS)
    & (proxy_df["engagement_rate"] < LOW_ENGAGEMENT_PCT)
)

proxy_df["opportunity_proxy"] = (
    proxy_df["ctr_opportunity"]
    | proxy_df["engagement_opportunity"]
)

print("This is a PROXY, not a causal or final target.")
print(f"Rows with valid position data: {len(proxy_df):,}")
print(
    f"Current opportunity-proxy rows: "
    f"{proxy_df['opportunity_proxy'].sum():,} "
    f"({proxy_df['opportunity_proxy'].mean():.1%})"
)

This is a PROXY, not a causal or final target.
Rows with valid position data: 28,795
Current opportunity-proxy rows: 6,267 (21.8%)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary success metric: Precision@50

Precision@50 measures how many of the first 50 pages recommended by the system correspond to the chosen observed outcome or validated opportunity definition.

I chose this metric because the purpose of the project is to prioritize limited human review capacity rather than classify every page perfectly.

I will consider the ML approach useful if its Precision@50 is at least **10 percentage points higher than the transparent rule-based baseline** on held-out data. I will use client-level holdout validation so pages from the same client do not appear in both training and testing.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
K = 50
REQUIRED_IMPROVEMENT = 0.10

print(f"Primary metric: Precision@{K}")
print(
    f"Success criterion: at least "
    f"{REQUIRED_IMPROVEMENT:.0%} absolute improvement "
    f"over the rule baseline."
)

print(
    f"Interpretation: among the first {K} pages sent for review, "
    "we want a larger share of useful candidates than the baseline produces."
)

Primary metric: Precision@50
Success criterion: at least 10% absolute improvement over the rule baseline.
Interpretation: among the first 50 pages sent for review, we want a larger share of useful candidates than the baseline produces.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis: one pseudonymized content item (one page).**

Each row represents one content item and contains observed search, traffic, engagement, and content-level measurements. The content and client IDs are pseudonymized identifiers used only for grouping, joining, and validation; they will not be treated as predictive features.

For this lane, the main working slice contains impressions, clicks, CTR, average position, position tier, sessions, engagement, age, and content type.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "position_tier",
    "sessions_90d",
    "engagement_rate",
    "content_age_days",
    "content_type",
]

lane_df = df[lane_columns].copy()

print(f"Lane dataframe shape: {lane_df.shape}")
print(f"Unique content items: {lane_df['content_id'].nunique():,}")
print(f"Rows: {len(lane_df):,}")

assert lane_df["content_id"].nunique() == len(lane_df), (
    "Expected one row per content item."
)

print("✓ Grain check passed: one row = one pseudonymized content item")

display(lane_df.head())

Lane dataframe shape: (30000, 11)
Unique content items: 30,000
Rows: 30,000
✓ Grain check passed: one row = one pseudonymized content item


,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,position_tier,sessions_90d,engagement_rate,content_age_days,content_type
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,10.6,striking,17,5.88,187,keyword article
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.05,20.3,page_3_5,9,0.00,445,keyword article
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,36.5,page_3_5,11,0.00,141,keyword article
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,6.2,page_1,78,1.28,463,keyword article
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.13,44.0,page_3_5,145,0.00,263,keyword article


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


A fixed rule is a useful baseline, but the opportunity pattern involves several interacting signals. CTR changes strongly with ranking position, while the reliability of CTR also depends on impression volume. Engagement depends on session volume and may vary by content type. Content age, search demand, position, CTR, and engagement may also interact.

A single rule such as "CTR below 1% means review" would therefore treat very different pages as if they were comparable.

ML may improve the ranking by learning combinations and nonlinear relationships among these observed signals. However, I will not assume that ML is automatically better. The fixed rule remains the baseline, and I will keep the ML approach only if it produces a meaningful improvement in held-out Precision@50 while remaining interpretable enough for human review.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show why one universal CTR threshold would be too simple.

analysis_df = df[
    (df["avg_position"] > 0)
    & (df["impressions_90d"] >= 100)
].copy()

ctr_by_position = (
    analysis_df
    .groupby("position_tier")["ctr"]
    .agg(["count", "median", "mean"])
    .round(3)
)

print("CTR differs across ranking-position tiers:")
display(ctr_by_position)

# Show that engagement context also differs by content type.
engagement_by_type = (
    df[df["sessions_90d"] >= 50]
    .groupby("content_type")["engagement_rate"]
    .agg(["count", "median", "mean"])
    .round(2)
)

print("\nEngagement also varies across content types:")
display(engagement_by_type)

print(
    "\nThese differences support testing a multi-signal ranking method "
    "against a simple fixed rule."
)

CTR differs across ranking-position tiers:


,count,median,mean
position_tier,,,
deep,879,0.00,0.055
page_1,8633,0.23,0.355
page_3_5,6058,0.06,0.142
striking,5903,0.15,0.256
top_3,533,0.19,0.334



Engagement also varies across content types:


,count,median,mean
content_type,,,
feedly article,25,1.72,3.02
keyword article,4894,1.82,2.71



These differences support testing a multi-signal ranking method against a simple fixed rule.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.